# 01 — Ingest: Problem Framing, Validation & Cleaning

**Case:** Telco Customer Churn (IBM Sample Dataset, via Kaggle)

**Scope of this notebook:** Problem Framing → Leakage Screening → Schema Validation → Cleaning → Train-Test Split

**Artifacts produced:**

| File | Contents |
| :--- | :--- |
| `train.parquet` | Clean training data, consumed by every downstream notebook |
| `test.parquet` | Clean held-out data — **do not mount this in notebooks 02–04** |
| `cost_assumptions.json` | Business cost parameters, re-read at evaluation time |
| `ingest_manifest.json` | Row counts, thresholds, and leakage-check results |

**Governing rule for this notebook:** no preprocessing decision may be influenced by the contents of the test set. The split happens before any statistics-driven manipulation, and the test file is deliberately left unattached to later notebooks so that this rule stops being something to remember and becomes something that cannot be violated.

---

## Setup

In [1]:
# Run once per session, then RESTART THE KERNEL before running the next cell.
# Python does not reload modules that are already imported, so without a restart
# this install silently has no effect.
#
# --force-reinstall --no-deps is required for the GitHub package: pip caches by
# version number, so a second install at the same version quietly keeps the old code.
%pip install -q --force-reinstall --no-deps git+https://github.com/rbennum/ml-utils.git
%pip install -q --force-reinstall --no-deps git+https://github.com/rbennum/telco-churn.git@main
%pip install -q pandera pyarrow

  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ml-utils is optional: the notebook still runs if it failed to install.
try:
    from ml_utils.utils import skim_data
    HAS_ML_UTILS = True
except ImportError:
    HAS_ML_UTILS = False
    print("ml_utils unavailable — falling back to a built-in summary.")

# pandera >= 0.24 moved the pandas API into its own submodule.
import pandera as pandera_root
try:
    import pandera.pandas as pa
except ImportError:
    pa = pandera_root

print(f"pandas {pd.__version__} | numpy {np.__version__} | pandera {pandera_root.__version__}")

pandas 2.3.3 | numpy 2.0.2 | pandera 0.32.1


In [3]:
# ---------------------------------------------------------------- configuration
# Move this block into src/<package>/config.py once it stabilises, then import it.

RANDOM_SEED = 29
TEST_SIZE = 0.2
TARGET = "Churn"

ON_KAGGLE = Path("/kaggle/working").exists()
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_PATHS = [
    Path("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    Path("../../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
]

print(f"ON_KAGGLE = {ON_KAGGLE}")
print(f"OUT_DIR   = {OUT_DIR.resolve()}")

ON_KAGGLE = True
OUT_DIR   = /kaggle/working


---

# Part 1 — Problem Framing

Everything in this section is written **before** looking at any model's performance. Choosing a metric after seeing results is a way of selecting for noise.

## 1.1 The Decision This Model Informs

A model that predicts churn is useless on its own. What is useful is a model that **decides which customers receive a retention offer**.

| Component | Definition |
| :--- | :--- |
| **Action** | Send a retention offer (time-limited discount) to selected customers |
| **Decision maker** | Customer Retention team |
| **Prediction moment** | Start of the monthly billing cycle, when the billing snapshot is available |
| **Decision unit** | One active customer per month |

**Prediction-time constraint.** Every feature in this dataset comes from billing and CRM systems, and their values exist at the moment the snapshot is taken — before churn status is known. No column is populated only after a customer leaves, such as a disconnection date, a stated reason for leaving, or a final refund amount. That claim is verified explicitly in Part 2.1 rather than assumed.

**Stopping condition.** Defined relative to the achievable ceiling, not as an invented percentage. Part 2.6 computes three points: the cost of the best naive policy, the cost of an **oracle** policy that knows exactly who will churn, and the gap between them — which is the entire improvement space available to any model.

The project continues if the model captures at least **40% of the oracle headroom**, and is abandoned below **15%**. Stating the target as a percentage reduction in total cost is an easy and serious mistake here: because offer effectiveness $\varepsilon$ bounds how many churners can actually be saved, even perfect targeting only cuts cost by roughly a fifth. A "15% total cost reduction" target sounds modest while in fact demanding near-perfect prediction.

### The Cost Model: Deriving the Profit-and-Loss Formula

A metric that cannot be derived from the consequences of being wrong is a convention, not a measurement. So the cost of each error type is fixed first, and the primary metric follows from it.

**Notation.** For customer $i$:

- $M_i$ — monthly bill (`MonthlyCharges`)
- $H_i$ — months of remaining tenure if the customer stays, which depends on contract type
- $g$ — gross margin on service revenue
- $\varepsilon$ — offer effectiveness: the probability that a would-be churner actually stays because of the offer
- $r$ — offer discount rate, as a proportion of the bill
- $d$ — discount duration in months
- $k$ — fixed contact cost per customer (agent time, channel cost)

**Value at risk** — the margin lost if the customer leaves:

$$V_i = M_i \times H_i \times g$$

**Offer cost** — a discount reduces revenue directly, so it erodes margin in full:

$$C_i = r \times M_i \times d + k$$

**Cost of each classification outcome.** Policy: send an offer when the model predicts churn.

| Outcome | What happened | Cost |
| :--- | :--- | :--- |
| TN | Stayed, not contacted | $0$ |
| FP | Stayed, contacted | $C_i$ |
| FN | Churned, not contacted | $V_i$ |
| TP | Churned, contacted | $(1-\varepsilon) V_i + C_i$ |

A true positive still carries cost, because offers do not always work: some customers leave despite the discount, and the discount is spent either way.

**Total loss function** for a policy $a_i \in \{0,1\}$:

$$\mathcal{L} = \sum_i \Big[ a_i \big( C_i + (1-\varepsilon) V_i y_i \big) + (1 - a_i) V_i y_i \Big]$$

**Optimal threshold.** For a customer with churn probability $p_i$, the expected cost of contacting is $p_i(1-\varepsilon)V_i + C_i$, and the expected cost of not contacting is $p_i V_i$. Contacting is cheaper exactly when:

$$p_i > p^*_i = \frac{C_i}{\varepsilon \, V_i} = \frac{r M_i d + k}{\varepsilon\, g\, M_i H_i}$$

Two consequences shape the rest of this project. First, **the threshold is neither 0.5 nor the output of F1 optimisation** — it falls directly out of the cost structure. Second, because $V_i$ and most of $C_i$ both scale with $M_i$, the threshold is nearly constant across customers; what makes it vary is the fixed contact cost $k$ and the contract-dependent horizon $H_i$. A low-bill customer on a short contract has to look far more likely to churn before contacting them pays off.

Because the decision uses the **value** of the probability rather than merely its rank, calibration is a requirement rather than a refinement — a model that ranks correctly but overstates probabilities will produce a wastefully aggressive contact policy.

In [4]:
# ------------------------------------------------- cost model assumptions
# Every number below is an ASSUMPTION that must be stated openly in the write-up,
# not a fact drawn from the dataset. The IBM dataset contains no costs, no margins,
# and no retention campaign outcomes. Sensitivity analysis over epsilon and margin
# runs in the evaluation notebook, because the business conclusion depends on them.

COST_ASSUMPTIONS = {
    "gross_margin": 0.40,          # g — gross margin on telecom service revenue
    "offer_discount_rate": 0.20,   # r — 20% off the monthly bill
    "offer_duration_months": 3,    # d — discount runs for 3 months
    "fixed_contact_cost": 2.00,    # k — USD, agent time plus channel cost per contact
    "offer_effectiveness": 0.30,   # epsilon — 30% of would-be churners are retained
    "horizon_by_contract": {       # H — months remaining if the customer stays
        "Month-to-month": 12,
        "One year": 18,
        "Two year": 24,
    },
    "currency": "USD",
    "note": (
        "Illustrative figures based on common telecom industry ranges, not internal "
        "company data. All financial conclusions are conditional on these assumptions."
    ),
}

with open(OUT_DIR / "cost_assumptions.json", "w") as f:
    json.dump(COST_ASSUMPTIONS, f, indent=2)

print(json.dumps(COST_ASSUMPTIONS, indent=2))

{
  "gross_margin": 0.4,
  "offer_discount_rate": 0.2,
  "offer_duration_months": 3,
  "fixed_contact_cost": 2.0,
  "offer_effectiveness": 0.3,
  "horizon_by_contract": {
    "Month-to-month": 12,
    "One year": 18,
    "Two year": 24
  },
  "currency": "USD",
  "note": "Illustrative figures based on common telecom industry ranges, not internal company data. All financial conclusions are conditional on these assumptions."
}


In [5]:
# ------------------------------------------------- cost model functions
# Strong candidates for src/<package>/business.py once proven, so that the
# evaluation notebook and the Streamlit app share one code path. Two code paths
# for the same logic is how training-serving skew starts.

def value_at_risk(monthly_charges, contract, a=COST_ASSUMPTIONS):
    """V_i — margin lost if this customer leaves."""
    horizon = pd.Series(contract).map(a["horizon_by_contract"]).to_numpy(dtype=float)
    return np.asarray(monthly_charges, dtype=float) * horizon * a["gross_margin"]


def offer_cost(monthly_charges, a=COST_ASSUMPTIONS):
    """C_i — cost of extending a retention offer."""
    mc = np.asarray(monthly_charges, dtype=float)
    return a["offer_discount_rate"] * mc * a["offer_duration_months"] + a["fixed_contact_cost"]


def optimal_threshold(monthly_charges, contract, a=COST_ASSUMPTIONS):
    """p*_i — the probability at which contacting becomes the cheaper action."""
    v = value_at_risk(monthly_charges, contract, a)
    c = offer_cost(monthly_charges, a)
    return np.clip(c / (a["offer_effectiveness"] * v), 0.0, 1.0)


def policy_cost(y_true, action, monthly_charges, contract, a=COST_ASSUMPTIONS):
    """Total cost of a contact policy. Lower is better."""
    y = np.asarray(y_true, dtype=float)
    act = np.asarray(action, dtype=float)
    v = value_at_risk(monthly_charges, contract, a)
    c = offer_cost(monthly_charges, a)
    contacted = act * (c + (1.0 - a["offer_effectiveness"]) * v * y)
    ignored = (1.0 - act) * v * y
    return float(np.sum(contacted + ignored))


def cost_per_customer(y_true, action, monthly_charges, contract, a=COST_ASSUMPTIONS):
    """The project's primary metric: expected cost per customer, in currency."""
    return policy_cost(y_true, action, monthly_charges, contract, a) / len(y_true)


print("Cost model functions ready.")

Cost model functions ready.


## 1.2 Predictive or Causal?

This question determines which half of the methodology applies, and churn is its textbook example.

**What this model answers:** who is likely to stop subscribing. That is purely predictive, and the standard supervised learning toolkit applies directly.

**What this model does not answer:** who will stay *because* they were contacted. That is a causal question, and accuracy on observational data contains no answer to it.

The distinction has a real price. The highest-risk customers are often the hardest to persuade — they have already decided to leave, and a discount delays the decision without changing it. Some customers churn *because* they were contacted, since a retention offer prompts them to reconsider a subscription that was running on autopilot. Targeting by predicted risk can therefore reduce net retention while the model's AUC looks excellent.

**The short test: does anyone act differently because of this prediction?** Yes — the retention team sends offers. So this is at least partly a causal problem, and purely predictive metrics will not detect its failure mode.

**Decision for this project.** What gets built is a predictive model, and that is stated openly as a limitation rather than disguised. The reason is structural: uplift estimation requires campaign outcome data — who was contacted, who was not, and what followed — and this dataset contains none of it. The parameter $\varepsilon$ in the cost model is where that causal assumption hides: it assumes offer effectiveness is uniform across customers, which is almost certainly false.

The consequence goes into the model card explicitly: **this model ranks risk; it does not establish that contacting high-risk customers is profitable.** The recommended deployment path is to run retention offers as a randomised experiment on a high-risk slice first, then use those outcomes to train a proper uplift model. Controlled experiments remain the gold standard; observational causal estimates are hypotheses awaiting confirmation.

## 1.3 Metric and Validation Commitments

Fixed now, before any numbers are seen.

| Role | Choice | Rationale |
| :--- | :--- | :--- |
| **Primary metric** | Expected cost per customer (USD) at the chosen threshold | The only number derivable from the cost structure in 1.1; makes model comparison a business statement rather than a leaderboard position |
| **Probability quality** | Log loss and Brier score | Proper scoring rules, threshold-free, minimised only when predicted probabilities equal the true conditional probabilities |
| **Ranking quality** | Average precision (PR-AUC) | Diagnostic for the queue-style use case; ROC-AUC is reported but not optimised |
| **Reporting diagnostics** | MCC, confusion matrix | MCC is better behaved than F1 under class imbalance |
| **CV scheme** | Stratified 5-Fold, `shuffle=True`, `random_state=42` | Standard for classification; keeps class proportions stable across folds |

**On F1.** F1 ignores true negatives entirely, moves with the threshold, and has no decision-theoretic derivation — it cannot be reconstructed from any cost structure. Since this project has quantified costs, selecting a model by F1 means discarding the most useful information available. F1 is still reported for comparability with other people's work, but never drives a decision.

**CV schemes deliberately not used.** `GroupKFold` is unnecessary because each row is a unique customer, verified in Part 2.1. `TimeSeriesSplit` with purging and embargo does not apply because this dataset is a single-point-in-time snapshot with no date column — which also means **out-of-time validation is impossible**, a limitation that belongs in the write-up.

---

# Part 2 — Data Acquisition, Validation & Cleaning

## 2.0 Loading the Raw Data

In [6]:
src = next((p for p in CANDIDATE_PATHS if p.exists()), None)

if src is None:
    raise FileNotFoundError(
        "Dataset not found. Pick one:\n"
        "  - Kaggle : Add Input -> search 'Telco Customer Churn' (blastchar)\n"
        "  - Local  : place the CSV at data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv\n"
        "             or run: kaggle datasets download -d blastchar/telco-customer-churn "
        "-p data/raw --unzip\n"
        f"Paths tried: {[str(p) for p in CANDIDATE_PATHS]}"
    )

df_raw = pd.read_csv(src)
print(f"Source : {src}")
print(f"Shape  : {df_raw.shape}")
df_raw.head()

Source : /kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape  : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
if HAS_ML_UTILS:
    display(skim_data(df_raw))
else:
    display(pd.DataFrame({
        "dtype": df_raw.dtypes.astype(str),
        "null_%": (df_raw.isna().mean() * 100).round(3),
        "n_unique": df_raw.nunique(),
        "sample_values": [list(df_raw[c].dropna().unique()[:5]) for c in df_raw.columns],
    }))
    print(f"Exact duplicate rows: {df_raw.duplicated().sum()} | shape: {df_raw.shape}")

Total duplicate rows: 0
DF shape: (7043, 21)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,customerID,object,0.0,-,-,-,-,7043,100.00,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOC..."
1,gender,object,0.0,-,-,-,-,2,0.03,"[Female, Male]"
2,SeniorCitizen,int64,0.0,0.0,83.785,0,1,2,0.03,"[0, 1]"
3,Partner,object,0.0,-,-,-,-,2,0.03,"[Yes, No]"
4,Dependents,object,0.0,-,-,-,-,2,0.03,"[No, Yes]"
5,tenure,int64,0.0,0.0,0.156,0,72,73,1.04,"[1, 34, 2, 45, 8]"
6,PhoneService,object,0.0,-,-,-,-,2,0.03,"[No, Yes]"
7,MultipleLines,object,0.0,-,-,-,-,3,0.04,"[No phone service, No, Yes]"
8,InternetService,object,0.0,-,-,-,-,3,0.04,"[DSL, Fiber optic, No]"
9,OnlineSecurity,object,0.0,-,-,-,-,3,0.04,"[No, Yes, No internet service]"


In [8]:
# Missing values in this dataset are not NaN — they are empty strings, so isna()
# never sees them. Any summary that reads only null_% will report clean data when
# the data is not clean.
blank_counts = {
    c: int((df_raw[c].astype(str).str.strip() == "").sum())
    for c in df_raw.select_dtypes(include="object").columns
}
blank_counts = {k: v for k, v in blank_counts.items() if v > 0}
print("Blank strings per column:", blank_counts if blank_counts else "none")

if blank_counts:
    col = next(iter(blank_counts))
    mask = df_raw[col].astype(str).str.strip() == ""
    display(df_raw.loc[mask, ["customerID", "tenure", "MonthlyCharges", col, "Contract", "Churn"]])

Blank strings per column: {'TotalCharges': 11}


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


## 2.1 Leakage Screening

Four checks run here. The fifth — the label permutation test — needs a complete pipeline and therefore runs in the baseline notebook, not this one.

**Post-outcome features.** Every column is tested against one question: is its value known only after churn has occurred? Columns such as a disconnection date, a stated reason for leaving, a final refund amount, or a closure ticket note would fail this test. This dataset contains none of them.

**Identifier leakage.** Row order, index values, and auto-incrementing IDs often correlate with the target for reasons that will not exist in production — for instance, an export sorted by signup date. This is tested, not assumed.

**Duplicates and near-duplicates across the split.** Exact duplicates inflate scores; near-duplicates are worse, because they survive a naive `drop_duplicates`.

In [9]:
# --- Identifier leakage: does customerID carry signal about the target?
tmp = df_raw.copy()
tmp["_row_order"] = np.arange(len(tmp))
tmp["_churn_bin"] = (tmp["Churn"] == "Yes").astype(int)
tmp["_id_numeric"] = pd.to_numeric(
    tmp["customerID"].str.extract(r"^(\d+)", expand=False), errors="coerce"
)

print("Row order vs churn correlation  :", round(tmp["_row_order"].corr(tmp["_churn_bin"]), 4))
print("ID prefix vs churn correlation  :", round(tmp["_id_numeric"].corr(tmp["_churn_bin"]), 4))
print("customerID is unique            :", tmp["customerID"].is_unique)
print("Rows per customer               :", round(len(tmp) / tmp["customerID"].nunique(), 3))

Row order vs churn correlation  : 0.0103
ID prefix vs churn correlation  : -0.0174
customerID is unique            : True
Rows per customer               : 1.0


In [10]:
# --- TotalCharges redundancy: is it just tenure x MonthlyCharges?
tc = pd.to_numeric(df_raw["TotalCharges"].astype(str).str.strip(), errors="coerce")
implied = df_raw["tenure"] * df_raw["MonthlyCharges"]
ok = tc.notna() & (tc > 0)

print(f"Correlation, TotalCharges vs tenure*MonthlyCharges : {tc[ok].corr(implied[ok]):.4f}")
print(f"Median absolute relative deviation                 : "
      f"{((tc[ok] - implied[ok]).abs() / tc[ok]).median():.2%}")

Correlation, TotalCharges vs tenure*MonthlyCharges : 0.9996
Median absolute relative deviation                 : 2.00%


In [11]:
# --- Duplicates, two separate checks.
# Exact duplicate rows are always zero while customerID is present, so that number
# is misleading if reported alone. The informative check is duplicates after the ID
# is set aside.
feat_cols = [c for c in df_raw.columns if c != "customerID"]

print(f"Duplicate customerIDs        : {df_raw['customerID'].duplicated().sum()}")
print(f"Exact duplicate rows         : {df_raw.duplicated().sum()}")
print(f"Duplicate profiles (no ID)   : {df_raw.duplicated(subset=feat_cols).sum()}")

dup_profiles = df_raw.loc[df_raw.duplicated(subset=feat_cols, keep=False), feat_cols]
if len(dup_profiles):
    display(dup_profiles.sort_values(feat_cols[:4]).head(8))

Duplicate customerIDs        : 0
Exact duplicate rows         : 0
Duplicate profiles (no ID)   : 22


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
542,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.55,19.55,No
1491,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.55,19.55,No
3499,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.90,20.9,Yes
4476,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.90,20.9,Yes
4495,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.10,70.1,Yes
4817,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.90,19.9,No
5170,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.65,19.65,No
5522,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes


### Screening decisions

| Column | Decision | Basis |
| :--- | :--- | :--- |
| `customerID` | Drop | A pure identifier. Dropped regardless of the correlation result, because the only way it can help is by memorising customers |
| `tenure` | Keep | Available in billing systems at scoring time. Not a future variable, even though its value stops exactly where the target is observed |
| `TotalCharges` | Keep, flagged | Almost perfectly collinear with `tenure` × `MonthlyCharges`. VIF is checked before the linear baseline, and SHAP output is read carefully, since strongly correlated features split credit in ways that are easy to misread |
| Everything else | Keep | Attributes that hold as of the snapshot |

**Duplicate profiles are kept.** Rows that are identical once `customerID` is removed are not a data error — they are different customers with the same profile, typically on the cheapest plan, short tenure, no add-on services. Removing them distorts the true distribution and erodes precisely the segment most prone to churn. Only duplicate `customerID` values are dropped, if any exist.

## 2.2 Data Types

`SeniorCitizen` is stored as 0/1 although it is semantically identical to the other Yes/No columns. Leaving it numeric would let scalers treat it as a continuous quantity and would make coefficients harder to read.

In [12]:
df = df_raw.copy()

# TotalCharges: object -> float. Blank strings become NaN, handled in 2.3.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")

# SeniorCitizen: 0/1 -> categorical, consistent with the other Yes/No columns.
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# Strip whitespace across all text columns.
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()

# Target -> binary. 1 = churn = positive class.
df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0}).astype("int8")
assert df[TARGET].isna().sum() == 0, "Some target values failed to map"

# Drop duplicate IDs, then the identifier itself.
before = len(df)
df = df.drop_duplicates(subset=["customerID"], keep="first").reset_index(drop=True)
df = df.drop(columns=["customerID"])

print(f"Rows  : {before} -> {len(df)}")
print(f"Shape : {df.shape}")
print(f"\nTarget balance:\n{df[TARGET].value_counts(normalize=True).round(4)}")

Rows  : 7043 -> 7043
Shape : (7043, 20)

Target balance:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


## 2.3 Missing Values

Understand the mechanism first, then pick the tool. Every blank `TotalCharges` belongs to a customer with `tenure == 0` — a new subscriber who has never been billed. The value is not missing; it is **structurally zero**.

Because the fill rule comes from domain knowledge (`tenure == 0` ⇒ `TotalCharges = 0`) rather than from any statistic computed on the data, applying it before the split leaks nothing: no information from the test set enters the decision. Median imputation is a different matter — that must be computed on training data only, which is why it belongs inside the pipeline in the baseline notebook, where it is refit on every fold.

In [13]:
na_mask = df["TotalCharges"].isna()
print(f"NaN TotalCharges                    : {na_mask.sum()}")
print(f"All of them have tenure == 0?       : {bool((df.loc[na_mask, 'tenure'] == 0).all())}")
print(f"tenure == 0 outside the NaN rows    : {int(((df['tenure'] == 0) & ~na_mask).sum())}")

if na_mask.any():
    display(df.loc[na_mask, ["tenure", "MonthlyCharges", "TotalCharges", "Contract", TARGET]])

NaN TotalCharges                    : 11
All of them have tenure == 0?       : True
tenure == 0 outside the NaN rows    : 0


,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,0,52.55,NaN,Two year,0
753,0,20.25,NaN,Two year,0
936,0,80.85,NaN,Two year,0
1082,0,25.75,NaN,Two year,0
1340,0,56.05,NaN,Two year,0
3331,0,19.85,NaN,Two year,0
3826,0,25.35,NaN,Two year,0
4380,0,20.00,NaN,Two year,0
5218,0,19.70,NaN,One year,0
6670,0,73.35,NaN,Two year,0


In [14]:
# Domain rule: deterministic, and independent of any statistic in the data.
df.loc[df["TotalCharges"].isna() & (df["tenure"] == 0), "TotalCharges"] = 0.0

remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print("Remaining missing values:", dict(remaining) if len(remaining) else "none")

Remaining missing values: none


## 2.4 Automated Schema Validation

Manual inspection catches problems once. A schema catches them on every run, including six months from now when an upstream source changes quietly.

The schema below belongs in `src/<package>/schema.py` so that every notebook validates on load. A schema failure is a stop condition, not a warning to be skimmed past.

In [15]:
YES_NO = ["Yes", "No"]
SERVICE_YES_NO = ["Yes", "No", "No internet service"]

churn_schema = pa.DataFrameSchema(
    {
        "gender": pa.Column(str, pa.Check.isin(["Male", "Female"])),
        "SeniorCitizen": pa.Column(str, pa.Check.isin(YES_NO)),
        "Partner": pa.Column(str, pa.Check.isin(YES_NO)),
        "Dependents": pa.Column(str, pa.Check.isin(YES_NO)),
        "tenure": pa.Column(int, pa.Check.in_range(0, 100)),
        "PhoneService": pa.Column(str, pa.Check.isin(YES_NO)),
        "MultipleLines": pa.Column(str, pa.Check.isin(["Yes", "No", "No phone service"])),
        "InternetService": pa.Column(str, pa.Check.isin(["DSL", "Fiber optic", "No"])),
        "OnlineSecurity": pa.Column(str, pa.Check.isin(SERVICE_YES_NO)),
        "OnlineBackup": pa.Column(str, pa.Check.isin(SERVICE_YES_NO)),
        "DeviceProtection": pa.Column(str, pa.Check.isin(SERVICE_YES_NO)),
        "TechSupport": pa.Column(str, pa.Check.isin(SERVICE_YES_NO)),
        "StreamingTV": pa.Column(str, pa.Check.isin(SERVICE_YES_NO)),
        "StreamingMovies": pa.Column(str, pa.Check.isin(SERVICE_YES_NO)),
        "Contract": pa.Column(
            str, pa.Check.isin(list(COST_ASSUMPTIONS["horizon_by_contract"]))
        ),
        "PaperlessBilling": pa.Column(str, pa.Check.isin(YES_NO)),
        "PaymentMethod": pa.Column(str),
        "MonthlyCharges": pa.Column(float, pa.Check.gt(0), nullable=False),
        "TotalCharges": pa.Column(float, pa.Check.ge(0), nullable=False),
        TARGET: pa.Column("int8", pa.Check.isin([0, 1])),
    },
    strict=True,       # an unexpected column is a failure, not a silent pass
    coerce=False,
)

# Contract categories are validated against the keys of horizon_by_contract, so a
# new contract type from upstream halts the pipeline instead of silently producing
# NaN inside the cost calculation.
df = churn_schema.validate(df, lazy=True)   # lazy=True reports every failure at once
print(f"Schema validation passed. Shape: {df.shape}")

Schema validation passed. Shape: (7043, 20)


## 2.5 Train-Test Split

The split happens before any manipulation that depends on data statistics. Scaling, statistical imputation, and encoding all happen after this point, inside a pipeline.

Stratification is on the target so churn proportions match on both sides. A time-based split is impossible with no date column; an entity-based split is unnecessary because one row is one customer.

In [16]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

train_df = X_train.copy()
train_df[TARGET] = y_train.to_numpy()
test_df = X_test.copy()
test_df[TARGET] = y_test.to_numpy()

print(f"Train: {train_df.shape} | Test: {test_df.shape}")
display(pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
    "full": y.value_counts(normalize=True),
}).round(4))

Train: (5634, 20) | Test: (1409, 20)


,train,test,full
Churn,,,
0,0.7346,0.7346,0.7346
1,0.2654,0.2654,0.2654


## 2.6 Trivial Baseline and Naive Policy Costs

Not logistic regression — the dumbest rule available. If a sophisticated model cannot beat this, nothing else in the project matters.

Naive contact policies are priced with the cost formula from Part 1.1, so there is a currency figure to beat from day one rather than an accuracy number to exceed. The **oracle** policy is included as the ceiling: it knows exactly who will churn and contacts only them, so no model can beat it.

**Everything here is computed on training data.** The test set stays untouched until the final evaluation notebook.

In [17]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# strategy="prior" predicts the base rate as a probability, not a hard label.
dummy = DummyClassifier(strategy="prior")
dummy_ll = -cross_val_score(dummy, X_train, y_train, cv=skf, scoring="neg_log_loss")

print(f"Churn base rate (train)  : {y_train.mean():.4f}")
print(f"Dummy log loss per fold  : {np.round(dummy_ll, 4)}")
print(f"Dummy log loss           : {dummy_ll.mean():.4f} (sd {dummy_ll.std():.4f})")
print("\nAny model that fails to beat this log loss has learned nothing.")

Churn base rate (train)  : 0.2654
Dummy log loss per fold  : [0.5785 0.5785 0.5785 0.5785 0.5788]
Dummy log loss           : 0.5786 (sd 0.0001)

Any model that fails to beat this log loss has learned nothing.


In [18]:
mc_tr = X_train["MonthlyCharges"].to_numpy()
ct_tr = X_train["Contract"].to_numpy()
n_tr = len(y_train)

policies = {
    "Contact nobody": np.zeros(n_tr),
    "Contact everyone": np.ones(n_tr),
    "Contact all month-to-month": (ct_tr == "Month-to-month").astype(float),
    # Oracle: knows exactly who churns and contacts only them. Unbeatable, so it
    # defines the ceiling.
    "ORACLE — contact exactly the churners": y_train.to_numpy(dtype=float),
}

rows = []
for name, action in policies.items():
    rows.append({
        "policy": name,
        "contacted_%": round(100 * action.mean(), 1),
        "cost_per_customer": round(cost_per_customer(y_train, action, mc_tr, ct_tr), 2),
    })

bench = pd.DataFrame(rows).sort_values("cost_per_customer").reset_index(drop=True)
display(bench)

ORACLE = float(bench.loc[bench.policy.str.startswith("ORACLE"), "cost_per_customer"].iloc[0])
BEST_NAIVE = float(bench.loc[~bench.policy.str.startswith("ORACLE"), "cost_per_customer"].min())
HEADROOM = BEST_NAIVE - ORACLE

print(f"\nBest naive policy       : USD {BEST_NAIVE:.2f} per customer")
print(f"Oracle ceiling          : USD {ORACLE:.2f} per customer")
print(f"Available headroom      : USD {HEADROOM:.2f} "
      f"({100 * HEADROOM / BEST_NAIVE:.1f}% of the benchmark)")
print(f"\nPass  (>=40% headroom)  : USD {BEST_NAIVE - 0.40 * HEADROOM:.2f} or lower")
print(f"Abort (<15% headroom)   : USD {BEST_NAIVE - 0.15 * HEADROOM:.2f} or higher")
print("\nEvery cost reduction any model can achieve is bounded by that headroom, "
      "because epsilon limits how many churners can actually be retained. Stating "
      "the target as a percentage of total cost would be misleading.")

,policy,contacted_%,cost_per_customer
0,ORACLE — contact exactly the churners,26.5,84.45
1,Contact all month-to-month,55.1,101.23
2,Contact nobody,0.0,102.90
3,Contact everyone,100.0,112.76



Best naive policy       : USD 101.23 per customer
Oracle ceiling          : USD 84.45 per customer
Available headroom      : USD 16.78 (16.6% of the benchmark)

Pass  (>=40% headroom)  : USD 94.52 or lower
Abort (<15% headroom)   : USD 98.71 or higher

Every cost reduction any model can achieve is bounded by that headroom, because epsilon limits how many churners can actually be retained. Stating the target as a percentage of total cost would be misleading.


In [19]:
# Per-customer optimal thresholds. Nearly constant, because value and cost both
# scale with MonthlyCharges; the variation comes from the fixed contact cost and
# the different horizons across contract types.
p_star = optimal_threshold(mc_tr, ct_tr)

display(
    pd.DataFrame({"Contract": ct_tr, "p_star": p_star})
    .groupby("Contract")["p_star"]
    .agg(["count", "mean", "min", "max"])
    .round(4)
)
print(f"\nMean threshold across all customers: {p_star.mean():.4f}")
print("Compare against the default 0.5, which rests on nothing at all.")

,count,mean,min,max
Contract,,,,
Month-to-month,3106,0.4445,0.4285,0.4905
One year,1164,0.2985,0.2856,0.3270
Two year,1364,0.2262,0.2142,0.2461



Mean threshold across all customers: 0.3615
Compare against the default 0.5, which rests on nothing at all.


### Offer Effectiveness Break-Even

The financial viability of this entire project rests on one parameter that does not exist in the dataset: $\varepsilon$, the probability that an offer actually retains a would-be churner. Contacting a certain churner only pays off when $\varepsilon V_i > C_i$, that is:

$$\varepsilon > \varepsilon^{\text{break-even}}_i = \frac{C_i}{V_i}$$

This is computed now rather than at the end, because if the true $\varepsilon$ sits below the break-even point, then no model however good makes the retention campaign profitable — and that is a far more important business finding than any model score.

In [20]:
v_tr = value_at_risk(mc_tr, ct_tr)
c_tr = offer_cost(mc_tr)
eps_breakeven = c_tr / v_tr

display(
    pd.DataFrame({"Contract": ct_tr, "breakeven_epsilon": eps_breakeven})
    .groupby("Contract")["breakeven_epsilon"]
    .agg(["count", "mean", "max"])
    .round(4)
)

eps_now = COST_ASSUMPTIONS["offer_effectiveness"]
share_viable = float((eps_breakeven < eps_now).mean())
print(f"\nCurrent epsilon assumption     : {eps_now:.2f}")
print(f"Customers worth contacting     : {share_viable:.1%} "
      f"(given certainty that they would churn)")
print(f"Highest break-even epsilon     : {eps_breakeven.max():.3f}")
print("\nIf an internal study puts epsilon below the break-even figure, stop the "
      "project here: the problem is the economics of the offer, not the model.")

,count,mean,max
Contract,,,
Month-to-month,3106,0.1333,0.1472
One year,1164,0.0896,0.0981
Two year,1364,0.0679,0.0738



Current epsilon assumption     : 0.30
Customers worth contacting     : 100.0% (given certainty that they would churn)
Highest break-even epsilon     : 0.147

If an internal study puts epsilon below the break-even figure, stop the project here: the problem is the economics of the offer, not the model.


## 2.7 Serialisation and Handoff

Parquet rather than CSV: it preserves dtypes, so the casting done once here does not quietly unravel every time a downstream notebook reads the file.

In [21]:
checks = {
    "no missing values in train": train_df.isna().sum().sum() == 0,
    "no missing values in test": test_df.isna().sum().sum() == 0,
    "train and test indices disjoint": not (set(X_train.index) & set(X_test.index)),
    "target is binary": set(train_df[TARGET].unique()) == {0, 1},
    "columns identical": list(train_df.columns) == list(test_df.columns),
    "row count preserved": len(train_df) + len(test_df) == len(df),
    "customerID dropped": "customerID" not in train_df.columns,
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "A check failed — do not continue to notebook 02."

PASS  no missing values in train
PASS  no missing values in test
PASS  train and test indices disjoint
PASS  target is binary
PASS  columns identical
PASS  row count preserved
PASS  customerID dropped


In [22]:
train_path = OUT_DIR / "train.parquet"
test_path = OUT_DIR / "test.parquet"

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

manifest = {
    "notebook": "01_ingest",
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "n_features": int(train_df.shape[1] - 1),
    "positive_rate_train": round(float(y_train.mean()), 5),
    "positive_rate_test": round(float(y_test.mean()), 5),
    "primary_metric": "expected_cost_per_customer_usd",
    "best_naive_policy_cost": round(BEST_NAIVE, 4),
    "oracle_policy_cost": round(ORACLE, 4),
    "available_headroom": round(HEADROOM, 4),
    "pass_threshold_cost": round(BEST_NAIVE - 0.40 * HEADROOM, 4),
    "abort_threshold_cost": round(BEST_NAIVE - 0.15 * HEADROOM, 4),
    "validation_scheme": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "oot_validation": "impossible — snapshot dataset with no time column",
    "leakage_checks": {
        "post_outcome_features": "none found",
        "identifier_dropped": "customerID",
        "duplicate_ids_removed": int(before - len(df)),
        "permutation_test": "deferred to notebook 03 (needs a full pipeline)",
    },
}
with open(OUT_DIR / "ingest_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"{train_path}  ->  {train_df.shape}")
print(f"{test_path}   ->  {test_df.shape}")
print(f"\n{json.dumps(manifest, indent=2)}")

/kaggle/working/train.parquet  ->  (5634, 20)
/kaggle/working/test.parquet   ->  (1409, 20)

{
  "notebook": "01_ingest",
  "random_seed": 29,
  "test_size": 0.2,
  "train_rows": 5634,
  "test_rows": 1409,
  "n_features": 19,
  "positive_rate_train": 0.26535,
  "positive_rate_test": 0.26544,
  "primary_metric": "expected_cost_per_customer_usd",
  "best_naive_policy_cost": 101.23,
  "oracle_policy_cost": 84.45,
  "available_headroom": 16.78,
  "pass_threshold_cost": 94.518,
  "abort_threshold_cost": 98.713,
  "validation_scheme": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
  "oot_validation": "impossible \u2014 snapshot dataset with no time column",
  "leakage_checks": {
    "post_outcome_features": "none found",
    "identifier_dropped": "customerID",
    "duplicate_ids_removed": 0,
    "permutation_test": "deferred to notebook 03 (needs a full pipeline)"
  }
}


In [23]:
# Record the environment as it actually is. On Kaggle, versions pinned in a
# requirements.txt do not apply when installs use --no-deps, because the notebook
# then runs against the image's bundled libraries.
!pip freeze > {OUT_DIR}/pip_freeze.txt
print(f"Environment recorded at {OUT_DIR}/pip_freeze.txt")

Environment recorded at /kaggle/working/pip_freeze.txt


---

## Summary

| Item | Result |
| :--- | :--- |
| Decision informed | Retention offer targeting, Customer Retention team, start of billing cycle |
| Cost of each error type | Quantified; FN loses remaining lifetime margin, FP spends offer cost |
| Problem nature | Predictive; the causal component is stated as a limitation, with a randomised experiment as the path forward |
| Trivial baseline | Dummy prior for log loss, three naive contact policies, plus the oracle ceiling |
| Stopping condition | A fraction of oracle headroom, computed from data rather than invented |
| Leakage screening | Post-outcome features, identifiers, duplicates, collinearity — done. Permutation test deferred to notebook 03 |
| Schema validation | Pandera, `strict=True`, failure halts the pipeline |
| Missing values | 11 structurally zero rows, domain rule, no data statistics involved |
| Split | Stratified 80/20, seed 42, before all statistical manipulation |
| Primary metric | Expected cost per customer (USD), derived from the cost structure |

**Limitations that must appear in the write-up and the model card**

1. There is no time dimension, so out-of-time validation is impossible and drift detection cannot be simulated.
2. The model is predictive, not causal. It ranks risk and does not establish that contacting high-risk customers is profitable.
3. All cost parameters are illustrative assumptions rather than internal company data. Financial conclusions are conditional and require sensitivity analysis.
4. Campaign viability rests on $\varepsilon$, which is absent from the dataset and measurable only through a randomised experiment. If it falls below the break-even figure in Part 2.6, no model rescues the project.

**Handoff to notebook 02**

Attach **only `train.parquet`**. `test.parquet` is deliberately left unmounted until the final evaluation notebook, which turns "do not touch the test set" from a rule into a physical constraint: the file simply is not on the filesystem during development.

Print `.shape` on both sides of every handoff. A mismatch tells you immediately that you are reading a stale artifact, before its metrics have a chance to mislead you.